In [1]:
import os
import sys
import time
import yaml
import pandas as pd
import numpy as np
import json
from string import Template

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import llm_tools as llt
from utils import is_casenum, extract_json

rng = np.random.default_rng()


In [2]:
MODEL = "claude-opus-4-6"
LLM_PROVIDER = "claude" # openai | claude
MAX_OUTPUT_TOKENS = 4000
input_cost = 2.50 # batch cost per 1M tokens
output_cost = 12.50 # batch cost per 1M tokens
output_tokens_per_request = 500
REQUESTS_PER_BATCH = 1000

In [3]:
MIN_IDX = 0
MAX_IDX = 150
REPLACE = False
REPLACE_OUTPUT_FILE = True
MODE = 'WRITE'  # ESTIMATE | BATCH | WRITE

In [4]:
meetings_df = pd.read_csv(os.path.join(DATA_PATH, "intermediate_data/cpc/meetings-manifest.csv"))
DATES = sorted(list(meetings_df['date']))

In [5]:
idx = rng.integers(low=MIN_IDX, high=MAX_IDX+1, size=1)
date = meetings_df.iloc[idx]['date'].values[0]
year = date[0:4]

agenda_filepath = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date, "agenda-cleaned-summarized.json")
agenda_override_filepath = os.path.join(LOCAL_PATH, "intermediate_data/cpc", year, date, "agenda-cleaned-summarized-override.json")
if os.path.exists(agenda_override_filepath):
    with open(agenda_override_filepath, 'r', encoding='utf-8') as f:
        agenda_json = json.load(f)
else:
    with open(agenda_filepath, 'r', encoding='utf-8') as f:
        agenda_json = json.load(f)

docs_filepath = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date, "supplemental-docs.pkl")
docs_df = pd.read_pickle(docs_filepath)

suppdocs_filepath = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date, "supplemental-docs-summarized.json")
with open(suppdocs_filepath, 'r') as f:
    suppdocs_json = json.load(f)

In [6]:
doc = rng.choice(suppdocs_json)
doc_id = doc['doc_id']
content = docs_df[docs_df['doc_id'] == doc_id]['content'].values[0]
agenda_text = ""
for agenda_item in agenda_json:
    if agenda_item['item_no'] in doc['referenced_agenda_items']:
        agenda_text += f"{agenda_item['text']}\n\n"

TEXT = f"""

DATE: {date}
DOC_ID: {doc_id}

SUMMARY:
{doc['summary']}

DOC_TYPE: {doc['document_type']}
AUTHOR_TYPE: {doc['author_type']}
REFERENCED_ITEMS: {str(doc['referenced_agenda_items'])}

SUPPORT_OR_OPPOSE:
{json.dumps(doc['support_or_oppose'], indent=2)}

=======================================================

DOCUMENT CONTENT:

{content}


========================================================

AGENDA CONTENT:

{agenda_text}

"""

with open("debug.txt", 'w', encoding='utf-8') as f:
    f.write(TEXT)

print(TEXT)



DATE: 2021-10-21
DOC_ID: 8

SUMMARY:
A letter from the Grace Lane Neighbors of Mount Saint Mary's University expressing support for the Mount Saint Mary's University Wellness Pavilion project, specifically Alternative 5 as described in the Final EIR. The neighbors highlight a legally binding agreement with the University to reduce peak-hour traffic below 2016 baseline levels and to create a closed campus that prohibits pedestrian entry, thereby eliminating overflow parking on neighborhood streets. The letter is signed by multiple homeowners on Grace Lane and urges the Commission to approve Alternative 5.

DOC_TYPE: PUBLIC COMMENT
AUTHOR_TYPE: INDIVIDUAL
REFERENCED_ITEMS: ['6', '7']

SUPPORT_OR_OPPOSE:
{
  "6": "DEFINITELY SUPPORT",
  "7": "DEFINITELY SUPPORT"
}


DOCUMENT CONTENT:


                                                                                    
  (cid:16)(cid:27)(cid:21)(cid:29)(cid:20)(cid:24)(cid:23)(cid:26)(cid:1)(cid:17)(cid:26)(cid:30)(cid:22)(cid:25)(cid:2